# Many Model Training — Sales Forecasting by Store

This notebook demonstrates the **Many Model Training (MMT)** pattern in Snowflake ML: train a separate XGBoost model for each store in parallel, wrap them in a single Custom Model, register to the Model Registry, and run batch inference via Snowpark.

**Sections**
1. Setup & cluster scaling
2. One-table data source (synthetic store sales)
3. Define the MMT training function
4. Train with `ManyModelTraining`
5. Inspect results & access per-store models
6. Restore a training run across sessions
7. Custom Model Wrapper — route inference to the right store model
8. Register to Model Registry
9. Batch inference via Snowpark Python
10. Batch inference via SQL
11. Large-scale SPCS batch inference (`run_batch`)

> **Requires:** Container Runtime notebook (MMT runs server-side on the Ray cluster).

---
## Configuration

Fill in the cell below once. Everything else in the notebook uses these variables.

In [ ]:
# ── Fill these in before running the notebook ─────────────────────────────────
DATABASE      = "ML_DEMOS"        # e.g. "ML_DEMOS"
SCHEMA        = "MMT"          # e.g. "MMT"
STAGE         = "STORE_MODELS_STAGE"   # Named stage for model artifacts
WAREHOUSE     = "ML_DEMO_WH"       # Warehouse for inference & SQL
COMPUTE_POOL  = "SYSTEM_COMPUTE_POOL_CPU"    # SPCS compute pool (Section 11 only)
REGISTRY_DB   = DATABASE              # Registry database (reuse same DB)
REGISTRY_SCH  = SCHEMA                # Registry schema
# ──────────────────────────────────────────────────────────────────────────────

TABLE_NAME      = f"{DATABASE}.{SCHEMA}.STORE_SALES"
INFERENCE_TABLE = f"{DATABASE}.{SCHEMA}.STORE_SALES_INFERENCE"
MODEL_NAME      = "STORE_SALES_PARTITIONED_MODEL"
MODEL_VERSION   = "v1"
TRAINING_RUN_ID = "store_sales_xgb_v1"

print("Config loaded.")
print(f"  Data table   : {TABLE_NAME}")
print(f"  Model stage  : @{DATABASE}.{SCHEMA}.{STAGE}")
print(f"  Registry     : {REGISTRY_DB}.{REGISTRY_SCH}")

---
## Section 1 — Setup & Cluster Scaling

In [ ]:
# Standard imports
import pandas as pd
import numpy as np

# Snowflake session (auto-available in Container Runtime notebooks)
from snowflake.snowpark.context import get_active_session
session = get_active_session()

# Set the default database/schema so we don't need to qualify every object
session.use_database(DATABASE)
session.use_schema(SCHEMA)

print(f"Connected: {session.get_current_database()}.{session.get_current_schema()}")

In [ ]:
# scale_cluster is not supported in this notebook environment.
# MMT will run on the single-node service — still works, just not distributed.
print("Skipping cluster scale-out (not supported for notebook origin). Proceeding with single node.")

---
## Section 2 — One-Table Data Source

The entire MMT workflow starts from **a single Snowflake table**. MMT will automatically split it into per-store partitions at training time — no pre-splitting required.

**Schema:**

| Column | Type | Description |
|--------|------|-------------|
| `STORE_ID` | STRING | Partition key — one model per store |
| `WEEK_OF_YEAR` | INT | 1–52 |
| `PROMO_FLAG` | INT | 1 = promotion active that week |
| `FOOT_TRAFFIC` | FLOAT | Avg daily visitors |
| `AVG_TEMP` | FLOAT | Average temperature (°F) |
| `IS_HOLIDAY` | INT | 1 = holiday week |
| `WEEKLY_SALES` | FLOAT | **Target** — total sales that week |

In [ ]:
# ── Create stage for model artifacts ──────────────────────────────────────────
session.sql(f"CREATE STAGE IF NOT EXISTS {DATABASE}.{SCHEMA}.{STAGE}").collect()

# ── Generate synthetic training data ──────────────────────────────────────────
np.random.seed(42)

stores = [f"STORE_{i:03d}" for i in range(1, 11)]   # 10 stores
weeks  = range(1, 53)                                # 52 weeks per store

rows = []
for store in stores:
    intercept    = np.random.uniform(5_000, 20_000)
    promo_effect = np.random.uniform(1_000, 4_000)
    traffic_coef = np.random.uniform(5, 15)
    temp_coef    = np.random.uniform(-50, 50)
    holiday_bump = np.random.uniform(2_000, 6_000)

    for week in weeks:
        promo      = np.random.binomial(1, 0.25)
        traffic    = np.random.normal(400, 80)
        temp       = 50 + 30 * np.sin((week - 13) * 2 * np.pi / 52) + np.random.normal(0, 5)
        is_holiday = 1 if week in (1, 22, 27, 47, 52) else 0
        sales = (
            intercept
            + promo_effect * promo
            + traffic_coef * traffic
            + temp_coef * temp
            + holiday_bump * is_holiday
            + np.random.normal(0, 500)
        )
        rows.append({
            "STORE_ID":     store,
            "WEEK_OF_YEAR": int(week),
            "PROMO_FLAG":   int(promo),
            "FOOT_TRAFFIC": round(float(traffic), 1),
            "AVG_TEMP":     round(float(temp), 1),
            "IS_HOLIDAY":   int(is_holiday),
            "WEEKLY_SALES": round(float(sales), 2),
        })

train_pd = pd.DataFrame(rows)
print(f"Generated {len(train_pd):,} rows across {train_pd['STORE_ID'].nunique()} stores")
train_pd.head()

In [ ]:
# Write to Snowflake — this is now our single source-of-truth training table
train_sp = session.create_dataframe(train_pd)
train_sp.write.mode("overwrite").save_as_table(TABLE_NAME)

print(f"Saved to {TABLE_NAME}")
session.table(TABLE_NAME).count()

In [ ]:
# Explore: rows per store — this is your partition size distribution
(
    session.table(TABLE_NAME)
    .group_by("STORE_ID")
    .count()
    .sort("STORE_ID")
    .to_pandas()
)

**Key point:** MMT reads from this one table and handles all the splitting. You never need to create per-store tables.

---
## Section 3 — Define the Training Function

MMT calls your training function **once per partition** with a fixed signature:

```
def my_train_fn(data_connector, context) -> model
```

| Argument | What it provides |
|----------|------------------|
| `data_connector` | Access to that partition's rows — call `.to_pandas()` to get a DataFrame |
| `context` | Metadata — `context.partition_id` gives you the store ID |
| **return value** | The trained model — MMT auto-serializes it to the stage |

In [ ]:
FEATURES = ["WEEK_OF_YEAR", "PROMO_FLAG", "FOOT_TRAFFIC", "AVG_TEMP", "IS_HOLIDAY"]
TARGET   = "WEEKLY_SALES"

def train_store_model(data_connector, context):
    """
    MMT training function — one call per store.

    Parameters
    ----------
    data_connector : DataConnector
        Provides rows for this store partition only.
    context : PartitionContext
        context.partition_id == the STORE_ID value for this partition.

    Returns
    -------
    XGBRegressor — auto-serialized by MMT (PickleSerde by default).
    """
    from xgboost import XGBRegressor

    df = data_connector.to_pandas()
    print(f"[{context.partition_id}] Training on {len(df)} rows")

    gridSearch

    X = df[FEATURES]
    y = df[TARGET]

    model = XGBRegressor(
        n_estimators=100,
        max_depth=4,
        learning_rate=0.1,
        random_state=42,
        n_jobs=-1,
    )
    model.fit(X, y)

    # Optionally log per-partition metrics
    preds = model.predict(X)
    rmse  = float(np.sqrt(np.mean((y - preds) ** 2)))
    print(f"[{context.partition_id}] Train RMSE: {rmse:.2f}")

    return model   # MMT serializes this automatically

---
## Section 4 — Train with `ManyModelTraining`

Three steps:
1. Create the trainer — link it to your function and a stage
2. Call `.run()` — specify the partition column and input DataFrame
3. Call `.wait()` — blocks until all partitions complete

In [ ]:
from snowflake.ml.modeling.distributors.many_model import ManyModelTraining
from snowflake.ml.modeling.distributors.distributed_partition_function.entities import RunStatus

FQ_STAGE = f"{DATABASE}.{SCHEMA}.{STAGE}"

trainer = ManyModelTraining(train_store_model, FQ_STAGE)

training_run = trainer.run(
    partition_by="STORE_ID",
    snowpark_dataframe=session.table(TABLE_NAME),
    run_id=TRAINING_RUN_ID,
)

final_status = training_run.wait()
print(f"\nTraining completed — overall status: {final_status}")

---
## Section 5 — Inspect Results & Access Per-Store Models

`training_run.partition_details` gives you the status of every partition. Use `training_run.get_model(store_id)` to retrieve a deserialized model object.

In [ ]:
# Summarize partition results
summary = [
    {"store_id": pid, "status": details.status}
    for pid, details in training_run.partition_details.items()
]
pd.DataFrame(summary).sort_values("store_id")

In [ ]:
# Retrieve and inspect one store's model
if final_status == RunStatus.SUCCESS:
    sample_store = "STORE_001"
    model_001 = training_run.get_model(sample_store)
    print(f"Model type  : {type(model_001).__name__}")
    print(f"n_estimators: {model_001.n_estimators}")
    print(f"Feature names: {model_001.feature_names_in_.tolist()}")

    # Quick sanity prediction — use the first row for that store
    sample_row = train_pd[train_pd["STORE_ID"] == sample_store][FEATURES].iloc[:1]
    pred = model_001.predict(sample_row)
    print(f"\nSample prediction for {sample_store}: ${pred[0]:,.2f}")

In [ ]:
# Collect all models into a dict — you'll reuse this in Section 7
partition_models = {
    store_id: training_run.get_model(store_id)
    for store_id in training_run.partition_details
}
print(f"Loaded {len(partition_models)} store models: {sorted(partition_models.keys())}")

---
## Section 6 — Restore a Training Run Across Sessions

Models are **persisted to the stage automatically**. You can restore a previous run in any future session — no retraining needed.

In [ ]:
from snowflake.ml.modeling.distributors.many_model import ManyModelRun

restored_run = ManyModelRun.restore_from(TRAINING_RUN_ID, FQ_STAGE)

restored_model = restored_run.get_model("STORE_005")
print(f"Restored model type: {type(restored_model).__name__}")

---
## Section 7 — Custom Model Wrapper

To register all store models together in the **Model Registry**, we wrap them in a single `CustomModel`. The `predict` method inspects the `STORE_ID` column and routes each batch to the correct per-store model.

**Pattern breakdown:**

| Component | Role |
|-----------|------|
| `ModelContext(models=partition_models)` | Packages all 10 store models into one serializable bundle |
| `self.context.model_ref(store_id)` | Retrieves a specific store's model at inference time |
| `@inference_api` + `function_type="TABLE_FUNCTION"` | Enables partitioned execution — Snowflake delivers rows one partition at a time |
| Caching (`_cached_partition_id`) | Avoids redundant `model_ref()` lookups when consecutive rows share the same store |

In [ ]:
import pandas as pd
from snowflake.ml.model.custom_model import CustomModel, ModelContext, inference_api


class PartitionedStoreModel(CustomModel):
    """
    One registered model that contains a sub-model for every store.

    At inference time Snowflake partitions the input table by STORE_ID and
    calls predict() once per partition — so input_df always belongs to a
    single store.  We read the store from the first row and look up the
    matching XGBoost model from context.
    """

    def __init__(self, context: ModelContext) -> None:
        super().__init__(context)
        self._cached_partition_id = None
        self._cached_model = None

    @inference_api
    def predict(self, input_df: pd.DataFrame) -> pd.DataFrame:
        store_id = str(input_df["STORE_ID"].iloc[0])

        if self._cached_partition_id != store_id:
            self._cached_model        = self.context.model_ref(store_id)
            self._cached_partition_id = store_id

        preds = self._cached_model.predict(input_df[FEATURES])
        return pd.DataFrame({
            "PREDICTED_SALES": preds,
        })


model_context    = ModelContext(models=partition_models)
partitioned_model = PartitionedStoreModel(context=model_context)

test_rows = train_pd[train_pd["STORE_ID"] == "STORE_001"].head(3).copy()
local_preds = partitioned_model.predict(test_rows)
print("Local smoke-test predictions:")
print(local_preds.to_string(index=False))

---
## Section 8 — Register to the Model Registry

Log the `PartitionedStoreModel` as a single model version. Setting `function_type="TABLE_FUNCTION"` tells the registry that inference should be dispatched with `PARTITION BY`.

In [ ]:
from snowflake.ml.registry import Registry

registry = Registry(
    session=session,
    database_name=REGISTRY_DB,
    schema_name=REGISTRY_SCH,
)

sample_input = train_pd[train_pd["STORE_ID"] == "STORE_001"][["STORE_ID"] + FEATURES].head(5)

mv = registry.log_model(
    model=partitioned_model,
    model_name=MODEL_NAME,
    version_name=MODEL_VERSION,
    sample_input_data=sample_input,
    options={
        "function_type": "TABLE_FUNCTION"
    },
    comment="XGBoost per-store sales forecast. 10 store sub-models trained with MMT.",
)

print(f"Registered: {mv.model_name} / {mv.version_name}")

---
## Section 9 — Batch Inference via Snowpark Python

`model_version.run()` executes inference on a **virtual warehouse**. Passing `partition_column="STORE_ID"` tells Snowflake to split the input and route each store's rows to the correct sub-model in parallel.

In [ ]:
# ── Create inference data ─────────────────────────────────────────────────────
# In production this would be a live or scheduled table.
# Here we generate a fresh week (week 53) for all stores.
inference_rows = []
for store in stores:
    inference_rows.append({
        "STORE_ID":     store,
        "WEEK_OF_YEAR": 53,
        "PROMO_FLAG":   1,
        "FOOT_TRAFFIC": round(float(np.random.normal(400, 80)), 1),
        "AVG_TEMP":     round(float(np.random.normal(35, 5)), 1),
        "IS_HOLIDAY":   0,
    })

inference_pd = pd.DataFrame(inference_rows)
inf_sp = session.create_dataframe(inference_pd)
inf_sp.write.mode("overwrite").save_as_table(INFERENCE_TABLE)

print(f"Inference table ready: {INFERENCE_TABLE}")
print(inference_pd.to_string(index=False))

In [ ]:
mv = registry.get_model(MODEL_NAME).version(MODEL_VERSION)

results = mv.run(
    session.table(INFERENCE_TABLE),
    function_name="PREDICT",
    partition_column="STORE_ID",
)

results_pd = results.to_pandas()
results_pd = results_pd.sort_values("STORE_ID").reset_index(drop=True)
print(f"Inference complete — {len(results_pd)} predictions")
results_pd

---
## Section 10 — Batch Inference via SQL

The same registered model is callable directly from SQL using the `PARTITION BY` clause. This is useful for embedding predictions in views, dynamic tables, or dbt models.

In [ ]:
sql = f"""
SELECT
    inf.STORE_ID,
    inf.WEEK_OF_YEAR,
    pred.PREDICTED_SALES
FROM {INFERENCE_TABLE} AS inf,
     TABLE(
         {REGISTRY_DB}.{REGISTRY_SCH}.{MODEL_NAME}!PREDICT(
             inf.STORE_ID,
             inf.WEEK_OF_YEAR,
             inf.PROMO_FLAG,
             inf.FOOT_TRAFFIC,
             inf.AVG_TEMP,
             inf.IS_HOLIDAY
         )
         OVER (PARTITION BY inf.STORE_ID)
     ) AS pred
ORDER BY inf.STORE_ID;
"""

sql_results = session.sql(sql).to_pandas()
print(f"SQL inference: {len(sql_results)} rows")
sql_results

---
## Section 11 — Large-Scale Batch Inference via SPCS (`run_batch`)

For very large datasets or GPU-accelerated models, use `run_batch()` instead of `run()`. This dispatches inference to an **SPCS compute pool** and writes results to a stage.

> **Requires:** `snowflake-ml-python >= 1.39.0` and a configured SPCS compute pool.

In [ ]:
from snowflake.ml.model.batch import InputSpec, OutputSpec

OUTPUT_STAGE = f"@{REGISTRY_DB}.{REGISTRY_SCH}.STORE_INFERENCE_RESULTS"
session.sql(f"CREATE OR REPLACE STAGE {REGISTRY_DB}.{REGISTRY_SCH}.STORE_INFERENCE_RESULTS").collect()

job = mv.run_batch(
    session.table(INFERENCE_TABLE),
    compute_pool=COMPUTE_POOL,
    input_spec=InputSpec(
        partition_column="STORE_ID"
    ),
    output_spec=OutputSpec(
        stage_location=OUTPUT_STAGE + "/predictions/"
    ),
)

print(f"Batch job submitted — ID: {job.id}")
print("Waiting for completion...")
job.wait()
print(f"Job status: {job.status}")

In [ ]:
import time
time.sleep(10)

session.sql(f"""
    CREATE OR REPLACE TEMPORARY TABLE {DATABASE}.{SCHEMA}.BATCH_RESULTS (
        PREDICTED_SALES DOUBLE
    )
""").collect()

session.sql(f"""
    COPY INTO {DATABASE}.{SCHEMA}.BATCH_RESULTS
    FROM {OUTPUT_STAGE}/predictions/
    FILE_FORMAT = (TYPE = 'PARQUET')
    MATCH_BY_COLUMN_NAME = CASE_INSENSITIVE
    ON_ERROR = CONTINUE
""").collect()

batch_results_pd = session.table(f"{DATABASE}.{SCHEMA}.BATCH_RESULTS").to_pandas()
print(f"Batch inference results: {len(batch_results_pd)} rows")
print(batch_results_pd)

---
## Summary

| Step | API / Class | Key point |
|------|-------------|-----------|
| Data | One Snowflake table | MMT splits it — no pre-partitioning needed |
| Train | `ManyModelTraining(fn, stage)` → `.run(partition_by=...)` | One XGBoost model per store, run in parallel |
| Access | `training_run.get_model(store_id)` | Deserialized model object, ready to use |
| Persist | `ManyModelRun.restore_from(run_id, stage)` | Re-load trained models in any future session |
| Wrap | `PartitionedStoreModel(CustomModel)` + `@inference_api` | Routes inference to the right store model via `model_ref()` |
| Register | `Registry.log_model(..., options={"function_type": "TABLE_FUNCTION"})` | All 10 store models in one registry entry |
| Infer (SQL) | `TABLE(model!PREDICT(...) OVER (PARTITION BY STORE_ID))` | Warehouse-parallel, SQL-native |
| Infer (Python) | `mv.run(df, function_name="PREDICT", partition_column="STORE_ID")` | Snowpark wrapper for the same call |
| Infer (SPCS) | `mv.run_batch(..., input_spec=InputSpec(partition_column=...))` | Compute-pool scale-out for large datasets |